In [2]:
import os
print(os.getcwd())
os.chdir('c:/Users/h/OneDrive/Documents/tfg/github')

c:\Users\h\OneDrive\Documents\tfg\github\notebooks


In [3]:
import cv2
from src.segmentation.medSAM_segmentor import MedSAMSegmentor
from src.utils.io import load_coco_annotations,find_image_path
from src.utils.visualization import get_overlay_mask
import torch
import os
import numpy as np

In [4]:
#config
CHECKPOINT = 'C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/models/medsam_vit_b.pth'
JSON_PATH = 'C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/result.json'
INPUT_ROOT = 'C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_DATASET'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda:0'

In [5]:

segmentor = MedSAMSegmentor(CHECKPOINT)
images_info, ann_by_image = load_coco_annotations(JSON_PATH)
IMAGE_SIZE= 1024

In [7]:
# images_info= images_info[61:62]
for img_info in images_info:
    img_id = img_info["id"]
    filename = img_info["file_name"]
    base = os.path.basename(filename)
    short_name = base.split('-', 1)[-1]



    image_path = find_image_path(INPUT_ROOT, short_name)
    if image_path is None:
        print(f"[SKIP] Image not found : {short_name}")
        continue

    image_bgr = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image_bgr is None:
        print(f"[SKIP] Could not read image : {image_path}")
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    anns = ann_by_image.get(img_id, [])
    if len(anns) == 0:
        print(f"[SKIP] {short_name} Has no annotations")
        continue

    mask = segmentor.segment_image( image_rgb,  anns)


    image_dir = os.path.dirname(image_path)
    masks_dir = os.path.join(image_dir, "MASKS")
    os.makedirs(masks_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    mask_path = os.path.join(masks_dir, base_name + "_mask.png")
    cv2.imwrite(mask_path, mask.astype(np.uint8) * 255)


    vis = get_overlay_mask(image_rgb, mask)

    vis_path = os.path.join(masks_dir, base_name + "_overlay.png")
    cv2.imwrite(vis_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    print(f"[OK] {short_name} -> {mask_path}")


[OK] 2a_2.png -> C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_DATASET\2a\section\MASKS\2a_2_mask.png
[OK] 2a_1802SE_2.jpg -> C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_DATASET\2a\section\MASKS\2a_1802SE_2_mask.png
[OK] 2a_1830SE.jpg -> C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_DATASET\2a\section\MASKS\2a_1830SE_mask.png
[OK] 2a_1878SE_2.jpg -> C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_DATASET\2a\section\MASKS\2a_1878SE_2_mask.png
[OK] 2a_1938SE.jpg -> C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_DATASET\2a\section\MASKS\2a_1938SE_mask.png
[OK] 2a_1939SE_2.jpg -> C:/Users/h/OneDrive/Documents/tfg/renal calculi classification/DATASET/SECTION_VIEW_DATASET/SECTION_SURFACE_